In [5]:
import re
import sqlite3
import pymysql
import datetime
from typing import Dict, List, Set, Any, Tuple, Optional
from collections import defaultdict
import random

# ===================== КОНФИГУРАЦИЯ ЗАПРОСОВ =====================

# Запросы для получения метаданных
MYSQL_METADATA_QUERIES = {
    'get_all_tables': """
        SELECT TABLE_NAME 
        FROM INFORMATION_SCHEMA.TABLES 
        WHERE TABLE_SCHEMA = %s
    """,
    
    'get_table_columns': """
        SELECT 
            COLUMN_NAME, 
            DATA_TYPE, 
            IS_NULLABLE, 
            COLUMN_KEY,
            EXTRA
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA = %s AND TABLE_NAME = %s
        ORDER BY ORDINAL_POSITION
    """,
    
    'get_primary_keys': """
        SELECT COLUMN_NAME
        FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
        WHERE TABLE_SCHEMA = %s 
          AND TABLE_NAME = %s
          AND CONSTRAINT_NAME = 'PRIMARY'
        ORDER BY ORDINAL_POSITION
    """
}

# ===================== УТИЛИТЫ =====================

def _convert_value_for_sqlite(value):
    """Преобразует значение в тип, поддерживаемый SQLite."""
    if value is None:
        return None
    
    # Обработка временных типов
    if isinstance(value, datetime.timedelta):
        total_seconds = int(value.total_seconds())
        hours = total_seconds // 3600
        minutes = (total_seconds % 3600) // 60
        seconds = total_seconds % 60
        microseconds = value.microseconds
        if microseconds:
            return f"{hours:02d}:{minutes:02d}:{seconds:02d}.{microseconds:06d}"
        return f"{hours:02d}:{minutes:02d}:{seconds:02d}"
    
    if isinstance(value, datetime.time):
        return value.strftime("%H:%M:%S.%f") if value.microsecond else value.strftime("%H:%M:%S")
    
    if isinstance(value, (datetime.date, datetime.datetime)):
        return value.isoformat()
    
    # Обработка бинарных данных
    if isinstance(value, bytes):
        try:
            return value.decode('utf-8', errors='replace')
        except:
            return str(value)
    
    # Обработка остальных типов
    if isinstance(value, (int, float, str, bool)):
        return value
    
    return str(value)

def _find_matching_table(name: str, all_tables: List[str]) -> Optional[str]:
    """Находит таблицу с совпадающим именем без учета регистра."""
    for table in all_tables:
        if table.lower() == name.lower():
            return table
    return None

def _normalize_query_for_db(query: str, all_tables: List[str], db_type: str = 'mysql') -> str:
    """Нормализует SQL-запрос для указанной БД."""
    # Удаляем комментарии
    query = re.sub(r'--.*?$|/\*.*?\*/', '', query, flags=re.DOTALL | re.MULTILINE)
    
    # Нормализация для MySQL
    if db_type == 'mysql':
        for table in all_tables:
            pattern = r'\b' + re.escape(table) + r'\b'
            query = re.sub(pattern, f'`{table}`', query, flags=re.IGNORECASE)
    
    # Нормализация для SQLite
    elif db_type == 'sqlite':
        query = query.replace('`', '"')
        query = re.sub(r'"\w+"\."(\w+)"', r'"\1"', query)
        
        for table in all_tables:
            pattern = r'\b' + re.escape(table) + r'\b'
            query = re.sub(pattern, f'"{table.lower()}"', query, flags=re.IGNORECASE)
        
        # Замена функций
        query = re.sub(r'NOW\(\)', 'CURRENT_TIMESTAMP', query, flags=re.IGNORECASE)
        query = re.sub(r'CURDATE\(\)', 'CURRENT_DATE', query, flags=re.IGNORECASE)
        query = re.sub(r'CURTIME\(\)', 'CURRENT_TIME', query, flags=re.IGNORECASE)
        query = re.sub(r'RAND\(\)', 'RANDOM()', query, flags=re.IGNORECASE)
    
    return query.strip()

def _extract_tables_from_query(query: str, all_tables: List[str]) -> List[str]:
    """Извлекает имена таблиц из SQL-запроса с точным сопоставлением."""
    potential_tables = set()
    
    # Регулярное выражение для поиска имен таблиц
    table_pattern = r'(?:FROM|JOIN)\s+([`"\w.]+)(?:\s+(?:AS\s+)?\w+)?'
    matches = re.findall(table_pattern, query, re.IGNORECASE)
    
    for match in matches:
        clean_name = match.strip('`"').split('.')[-1]
        potential_tables.add(clean_name)
    
    # Поиск в подзапросах
    subquery_pattern = r'\(\s*SELECT.*?FROM\s+([`"\w.]+)'
    matches = re.findall(subquery_pattern, query, re.IGNORECASE | re.DOTALL)
    for match in matches:
        clean_name = match.strip('`"').split('.')[-1]
        potential_tables.add(clean_name)
    
    # Сопоставляем с реальными таблицами базы
    found_tables = set()
    for potential in potential_tables:
        matched_table = _find_matching_table(potential, all_tables)
        if matched_table:
            found_tables.add(matched_table)
    
    return list(found_tables)

# ===================== ФУНКЦИИ РАБОТЫ С МЕТАДАННЫМИ =====================

def get_all_table_names(cursor, metadata_queries: Dict[str, str], database: str) -> List[str]:
    """Получает список всех таблиц в базе данных."""
    cursor.execute(metadata_queries['get_all_tables'], (database,))
    return [row['TABLE_NAME'] for row in cursor.fetchall()]

def get_table_schemas(cursor, table_names: List[str], all_tables: List[str], 
                     metadata_queries: Dict[str, str], database: str) -> Dict[str, Dict]:
    """Получает схемы таблиц."""
    schemas = {}
    
    for table in table_names:
        if not table:
            continue
            
        if table not in all_tables:
            matched_table = _find_matching_table(table, all_tables)
            if matched_table:
                table = matched_table
            else:
                continue
        
        # Получаем информацию о столбцах
        cursor.execute(metadata_queries['get_table_columns'], (database, table))
        columns = cursor.fetchall()
        
        if not columns:
            continue
        
        # Получаем информацию о первичных ключах
        cursor.execute(metadata_queries['get_primary_keys'], (database, table))
        pk_columns = {row['COLUMN_NAME'] for row in cursor.fetchall()}
        
        schemas[table] = {
            'columns': columns,
            'primary_keys': pk_columns
        }
    
    return schemas

# ===================== ФУНКЦИИ РАБОТЫ С ДАННЫМИ =====================

def execute_query(cursor, query: str, all_tables: List[str]) -> List[Dict]:
    """Выполняет SQL-запрос и возвращает результаты."""
    normalized_query = _normalize_query_for_db(query, all_tables, 'mysql')
    cursor.execute(normalized_query)
    return cursor.fetchall()

def fetch_table_data(cursor, table_name: str, schema: Dict, 
                    pk_values: Set, actual_row_limit: int) -> List[Dict]:  # Изменён параметр row_limit -> actual_row_limit
    """Получает данные таблицы с приоритетом строк по PK и дополнением до лимита."""
    pk_columns = list(schema['primary_keys'])
    columns = [f"`{col['COLUMN_NAME']}`" for col in schema['columns']]
    
    rows = []
    if pk_values and pk_columns:
        pk_col = pk_columns[0]
        format_str = ','.join(['%s'] * len(pk_values))
        query = f"""
            SELECT {', '.join(columns)}
            FROM `{table_name}`
            WHERE `{pk_col}` IN ({format_str})
            LIMIT {actual_row_limit}  # Используем actual_row_limit
        """
        try:
            cursor.execute(query, tuple(pk_values))
            rows = cursor.fetchall()
        except (pymysql.Error, KeyError):
            pass
    
    if len(rows) < actual_row_limit:  # Используем actual_row_limit
        additional_limit = actual_row_limit - len(rows)  # Используем actual_row_limit
        where_clause = ""
        params = []
        
        if pk_columns and rows:
            pk_col = pk_columns[0]
            exclude_ids = [
                row[pk_col] for row in rows 
                if pk_col in row and row[pk_col] is not None
            ]
            if exclude_ids:
                format_str = ','.join(['%s'] * len(exclude_ids))
                where_clause = f"WHERE `{pk_col}` NOT IN ({format_str})"
                params = exclude_ids
        
        query = f"""
            SELECT {', '.join(columns)}
            FROM `{table_name}`
            {where_clause}
            LIMIT {additional_limit}
        """
        cursor.execute(query, tuple(params))
        additional_rows = cursor.fetchall()
        rows.extend(additional_rows)
    
    return rows[:actual_row_limit]  # Используем actual_row_limit

def fetch_random_table_data(cursor, table_name: str, schema: Dict, 
                           actual_row_limit: int) -> List[Dict]:  # Изменён параметр row_limit -> actual_row_limit
    """Получает случайные строки из таблицы."""
    columns = [f"`{col['COLUMN_NAME']}`" for col in schema['columns']]
    
    try:
        query = f"""
            SELECT {', '.join(columns)}
            FROM `{table_name}`
            ORDER BY RAND()
            LIMIT {actual_row_limit}  # Используем actual_row_limit
        """
        cursor.execute(query)
        return cursor.fetchall()
    except pymysql.Error:
        # Если ORDER BY RAND() не поддерживается, берем первые N строк
        query = f"""
            SELECT {', '.join(columns)}
            FROM `{table_name}`
            LIMIT {actual_row_limit}  # Используем actual_row_limit
        """
        cursor.execute(query)
        return cursor.fetchall()

def collect_pk_values(detail_rows: List[Dict], table_schemas: Dict[str, Dict]) -> Dict[str, Set]:
    """Собирает значения первичных ключей из результата запроса."""
    pk_values_map = defaultdict(set)
    
    for row in detail_rows:
        for table_name, schema in table_schemas.items():
            pk_columns = schema['primary_keys']
            if not pk_columns:
                continue
            
            table_prefix = table_name.lower()
            
            for pk_col in pk_columns:
                candidates = [
                    pk_col,
                    f"{table_prefix}.{pk_col}",
                    f"{table_name}.{pk_col}",
                    pk_col.lower(),
                    f"{table_prefix}.{pk_col.lower()}"
                ]
                
                for candidate in candidates:
                    if candidate in row and row[candidate] is not None:
                        pk_values_map[table_name].add(row[candidate])
                        break
    
    return pk_values_map

# ===================== ФУНКЦИИ РАБОТЫ С SQLite =====================

def create_sqlite_database(db_path: str, schemas: Dict[str, Dict], 
                          table_data: Dict[str, List[Dict]]) -> None:
    """Создает SQLite-базу и наполняет данными."""
    type_mapping = {
        'int': 'INTEGER',
        'tinyint': 'INTEGER',
        'smallint': 'INTEGER',
        'mediumint': 'INTEGER',
        'bigint': 'INTEGER',
        'float': 'REAL',
        'double': 'REAL',
        'decimal': 'REAL',
        'numeric': 'REAL',
        'date': 'TEXT',
        'datetime': 'TEXT',
        'timestamp': 'TEXT',
        'time': 'TEXT',
        'year': 'INTEGER',
        'char': 'TEXT',
        'varchar': 'TEXT',
        'text': 'TEXT',
        'blob': 'BLOB',
        'enum': 'TEXT',
        'set': 'TEXT'
    }
    
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        cursor.execute("PRAGMA foreign_keys = ON;")
        
        sqlite_table_names = {}
        
        for original_table_name, schema in schemas.items():
            if not schema['columns']:
                continue
                
            sqlite_table_name = original_table_name.lower()
            sqlite_table_names[original_table_name] = sqlite_table_name
            
            columns_def = []
            pk_constraints = []
            
            for col in schema['columns']:
                col_name = col['COLUMN_NAME']
                mysql_type = col['DATA_TYPE'].lower()
                sqlite_type = 'TEXT'
                
                for key, value in type_mapping.items():
                    if key in mysql_type:
                        sqlite_type = value
                        break
                
                is_auto_increment = 'auto_increment' in col['EXTRA'].lower()
                nullable = '' if col['IS_NULLABLE'] == 'YES' else 'NOT NULL'
                
                if is_auto_increment and 'INTEGER' in sqlite_type:
                    column_def = f'"{col_name.lower()}" {sqlite_type} PRIMARY KEY AUTOINCREMENT'
                else:
                    column_def = f'"{col_name.lower()}" {sqlite_type} {nullable}'
                    if col_name in schema['primary_keys']:
                        pk_constraints.append(f'"{col_name.lower()}"')
                
                columns_def.append(column_def)
            
            if pk_constraints:
                pk_str = ", ".join(pk_constraints)
                columns_def.append(f"PRIMARY KEY ({pk_str})")
            
            create_query = f'CREATE TABLE IF NOT EXISTS "{sqlite_table_name}" ({", ".join(columns_def)})'
            try:
                cursor.execute(create_query)
            except sqlite3.OperationalError:
                continue
        
        # Вставляем данные
        for original_table_name, rows in table_data.items():
            if not rows or original_table_name not in sqlite_table_names:
                continue
                
            sqlite_table_name = sqlite_table_names[original_table_name]
            columns = list(rows[0].keys())
            columns_lower = [col.lower() for col in columns]
            
            try:
                cursor.execute(f"PRAGMA table_info(\"{sqlite_table_name}\")")
                existing_columns = {row[1].lower() for row in cursor.fetchall()}
            except sqlite3.OperationalError:
                continue
            
            valid_columns = [col for col in columns_lower if col in existing_columns]
            
            if not valid_columns:
                continue
            
            placeholders = ", ".join(["?"] * len(valid_columns))
            insert_query = f"""
                INSERT OR IGNORE INTO "{sqlite_table_name}" 
                ({', '.join(f'"{col}"' for col in valid_columns)}) 
                VALUES ({placeholders})
            """
            
            for row in rows:
                values = []
                for orig_col in columns:
                    lower_col = orig_col.lower()
                    if lower_col in existing_columns:
                        value = row.get(orig_col)
                        if value is None and lower_col in row:
                            value = row.get(lower_col)
                        values.append(_convert_value_for_sqlite(value))
                
                try:
                    cursor.execute(insert_query, values)
                except (sqlite3.IntegrityError, sqlite3.OperationalError):
                    pass
        
        conn.commit()

def validate_query_in_sqlite(db_path: str, query: str, all_tables: List[str]) -> Tuple[bool, Any]:
    """Выполняет запрос в SQLite и возвращает результат."""
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        normalized_query = _normalize_query_for_db(query, all_tables, 'sqlite')
        
        try:
            cursor.execute(normalized_query)
            result = cursor.fetchone()
            return True, result
        except sqlite3.OperationalError as e:
            return False, str(e)

# ===================== ОСНОВНАЯ ФУНКЦИЯ =====================

def create_local_db_snapshot(
    remote_config: Dict[str, Any],
    aggregation_query: str,
    detail_query: str,
    metadata_queries: Dict[str, str] = None,
    output_db_path: str = "local_snapshot.db",
    row_limit: int = 40,
    essential_tables: List[str] = None
) -> None:
    """
    Создает локальную SQLite-копию удаленной MySQL-базы.
    
    Args:
        remote_config: Конфигурация подключения к MySQL
        aggregation_query: Агрегирующий запрос для проверки
        detail_query: Детальный запрос для получения данных
        metadata_queries: Словарь с запросами для получения метаданных
        output_db_path: Путь к выходному SQLite файлу
        row_limit: Лимит строк на таблицу
        essential_tables: Список обязательных таблиц
    """
    if metadata_queries is None:
        metadata_queries = MYSQL_METADATA_QUERIES
    
    if essential_tables is None:
        essential_tables = []
    
    # Устанавливаем минимальное количество строк
    MIN_ROWS = 20
    # Гарантируем, что максимальный лимит не меньше минимального
    MAX_ROWS = max(MIN_ROWS, row_limit)
    
    print("\033[94m" + "="*60)
    print("Создание локального снапшота БД")
    print("="*60 + "\033[0m")
    
    # Подключаемся к удаленной БД
    with pymysql.connect(**remote_config) as remote_conn:
        with remote_conn.cursor(pymysql.cursors.DictCursor) as cursor:
            
            # Шаг 1: Получаем список всех таблиц
            all_tables = get_all_table_names(
                cursor, metadata_queries, remote_config['database']
            )
            print(f"Всего таблиц в базе: {len(all_tables)}")
            
            # Шаг 2: Анализ зависимостей таблиц из запросов
            dependent_tables = set()
            dependent_tables.update(_extract_tables_from_query(detail_query, all_tables))
            dependent_tables.update(_extract_tables_from_query(aggregation_query, all_tables))
            
            # Добавляем обязательные таблицы
            for table in essential_tables:
                matched_table = _find_matching_table(table, all_tables)
                if matched_table:
                    dependent_tables.add(matched_table)
            
            dependent_tables = list(dependent_tables)
            print(f"Таблицы для снапшота: {len(dependent_tables)}")
            
            if not dependent_tables:
                raise ValueError("Не удалось определить таблицы для снапшота")
            
            # Шаг 3: Получаем схемы таблиц
            table_schemas = get_table_schemas(
                cursor, dependent_tables, all_tables, 
                metadata_queries, remote_config['database']
            )
            
            # Шаг 4: Выполняем детальный запрос
            print("\nВыполнение детального запроса...")
            detail_rows = execute_query(cursor, detail_query, all_tables)
            print(f"Получено {len(detail_rows)} строк")
            
            if not detail_rows:
                raise ValueError("Детальный запрос вернул пустой результат")
            
            # Шаг 5: Собираем значения PK
            pk_values_map = collect_pk_values(detail_rows, table_schemas)
            
            # Шаг 6: Получаем данные для зависимых таблиц
            table_data = {}
            for table_name in dependent_tables:
                if table_name in table_schemas:
                    # Генерируем случайное количество строк для этой таблицы
                    actual_row_limit = random.randint(MIN_ROWS, MAX_ROWS)
                    table_data[table_name] = fetch_table_data(
                        cursor, table_name, table_schemas[table_name],
                        pk_values_map.get(table_name, set()), actual_row_limit  # Передаём actual_row_limit
                    )
                    print(f"Загружено {len(table_data[table_name])} строк из {table_name} (лимит: {actual_row_limit})")
            
            # Шаг 7: Получаем данные для остальных таблиц
            remaining_tables = [t for t in all_tables if t not in dependent_tables]
            
            if remaining_tables:
                remaining_schemas = get_table_schemas(
                    cursor, remaining_tables, all_tables,
                    metadata_queries, remote_config['database']
                )
                table_schemas.update(remaining_schemas)
                
                for table_name in remaining_tables:
                    if table_name in remaining_schemas:
                        # Генерируем случайное количество строк для этой таблицы
                        actual_row_limit = random.randint(MIN_ROWS, MAX_ROWS)
                        table_data[table_name] = fetch_random_table_data(
                            cursor, table_name, remaining_schemas[table_name], actual_row_limit  # Передаём actual_row_limit
                        )
                        print(f"Загружено {len(table_data[table_name])} строк из {table_name} (лимит: {actual_row_limit})")

    
    # Шаг 8: Создаем SQLite базу
    print(f"\nСоздание локальной базы: {output_db_path}")
    create_sqlite_database(output_db_path, table_schemas, table_data)
    
    # Шаг 9: Проверяем агрегирующий запрос
    print("\nПроверка агрегирующего запроса в SQLite...")
    success, result = validate_query_in_sqlite(
        output_db_path, aggregation_query, all_tables
    )
    
    if success:
        print(f"\033[92mЗапрос успешно выполнен. Результат: {result}\033[0m")
    else:
        print(f"\033[93mОшибка выполнения запроса: {result}\033[0m")
    
    print(f"\n\033[92mСнапшот сохранен в {output_db_path}\033[0m")

# ===================== ПРИМЕР ИСПОЛЬЗОВАНИЯ =====================

if __name__ == "__main__":
    # Конфигурация подключения
    remote_config = {
        'host': 'relational.fel.cvut.cz',
        'port': 3306,
        'user': 'guest',
        'password': 'ctu-relational',
        'database': 'Credit',
        'charset': 'utf8mb4',
        'cursorclass': pymysql.cursors.DictCursor
    }
    
    # Пользовательские запросы
    aggregation_query = """
    SELECT 
    COUNT(*) AS num_categories_above_avg_charge
    FROM (
        SELECT 
            cat.category_no
        FROM 
            charge ch
        JOIN 
            category cat ON ch.category_no = cat.category_no
        GROUP BY 
            cat.category_no
        HAVING 
            SUM(ch.charge_amt) > (
                SELECT AVG(total_per_cat) 
                FROM (
                    SELECT SUM(charge_amt) AS total_per_cat
                    FROM charge
                    GROUP BY category_no
                )
            )
    ) AS filtered_categories;
    """
    
    detail_query = """
    SELECT 
        ch.*,
        cat.*
    FROM 
        charge ch
    JOIN 
        category cat ON ch.category_no = cat.category_no;
    """
    
    # Дополнительные настройки
    essential_tables = ['constructorResults', 'constructors', 'races', 'circuits']
    
    # Создание снапшота
    create_local_db_snapshot(
        remote_config=remote_config,
        aggregation_query=aggregation_query,
        detail_query=detail_query,
        output_db_path="local_snapshot.db",
        row_limit=90,
    )

Создание локального снапшота БД
Всего таблиц в базе: 8
Таблицы для снапшота: 2

Выполнение детального запроса...
Получено 1600000 строк
Загружено 10 строк из category (лимит: 75)
Загружено 54 строк из charge (лимит: 54)
Загружено 66 строк из member (лимит: 66)
Загружено 9 строк из region (лимит: 77)
Загружено 21 строк из corporation (лимит: 21)
Загружено 85 строк из payment (лимит: 85)
Загружено 64 строк из provider (лимит: 64)
Загружено 70 строк из statement (лимит: 70)

Создание локальной базы: local_snapshot.db

Проверка агрегирующего запроса в SQLite...
Запрос успешно выполнен. Результат: (5,)

Снапшот сохранен в local_snapshot.db


In [13]:
import sqlite3
import pandas as pd
from io import StringIO
import csv

def get_database_info_and_csv():
    # Подключаемся к локальной базе данных
    conn = sqlite3.connect('local_credit_database.db')
    cursor = conn.cursor()

    # Получаем список таблиц
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    output = StringIO()
    writer = csv.writer(output)

    for table_name in tables:
        table_name = table_name[0]
        print(f"Обработка таблицы: {table_name}")

        # Получаем структуру таблицы (CREATE TABLE)
        cursor.execute(f"SELECT sql FROM sqlite_master WHERE type='table' AND name='{table_name}';")
        create_table_sql = cursor.fetchone()[0]
        output.write(f"\n--- Структура таблицы {table_name} ---\n")
        output.write(f"{create_table_sql}\n")

        # Получаем и выводим все данные из таблицы
        df = pd.read_sql_query(f"SELECT * FROM {table_name};", conn)
        output.write(f"\n--- Данные таблицы {table_name} ---\n")
        
        # Записываем заголовки
        writer.writerow(df.columns.tolist())
        # Записываем данные
        for row in df.itertuples(index=False, name=None):
            writer.writerow(row)
        output.write("\n")  # Добавляем пустую строку между таблицами

    conn.close()
    
    return output.getvalue()

# Вызов функции и вывод результата
result = get_database_info_and_csv()
print(result)

Обработка таблицы: region
Обработка таблицы: category
Обработка таблицы: corporation
Обработка таблицы: provider
Обработка таблицы: member
Обработка таблицы: statement
Обработка таблицы: charge
Обработка таблицы: payment

--- Структура таблицы region ---
CREATE TABLE region (
                region_no INTEGER PRIMARY KEY,
                region_name TEXT NOT NULL,
                street TEXT NOT NULL,
                city TEXT NOT NULL,
                state_prov TEXT NOT NULL,
                country TEXT NOT NULL,
                mail_code TEXT NOT NULL,
                phone_no TEXT NOT NULL,
                region_code TEXT NOT NULL
            )

--- Данные таблицы region ---
region_no,region_name,street,city,state_prov,country,mail_code,phone_no,region_code
1,North American,111 First St.,Toronto,ON,Ca,,,
2,South American,222 Second St.,Sao Paulo,,Br,,,
3,Scandanavian,333 Third St.,Goteborg,,Sw,,,
4,Western Europea,444 Fourth St.,Brussels,,,,,
5,Eastern Europea,555 Fifth St St,Bud

In [15]:
import sqlite3
import pandas as pd

def execute_sql_query(db_file, query):
    """
    Выполняет SQL-запрос к локальной базе данных и возвращает результат в виде DataFrame.

    :param db_file: путь к файлу локальной базы данных SQLite
    :param query: SQL-запрос для выполнения
    :return: результат запроса в виде pandas DataFrame
    """
    # Подключаемся к базе данных
    conn = sqlite3.connect(db_file)
    
    try:
        # Выполняем запрос и возвращаем результат в виде DataFrame
        result_df = pd.read_sql_query(query, conn)
        return result_df
    except Exception as e:
        print(f"Произошла ошибка при выполнении запроса: {e}")
        return None
    finally:
        # Закрываем соединение
        conn.close()

# Пример использования:
# db_file = 'local_credit_database.db'
# query = 'SELECT * FROM member LIMIT 10;'
# result = execute_sql_query(db_file, query)
# print(result)

In [36]:
query = """
SELECT 
    COUNT(*) AS num_categories_above_avg_charge
FROM (
    SELECT 
        cat.category_no
    FROM 
        charge ch
    JOIN 
        category cat ON ch.category_no = cat.category_no
    GROUP BY 
        cat.category_no
    HAVING 
        SUM(ch.charge_amt) > (
            SELECT AVG(total_per_cat) 
            FROM (
                SELECT SUM(charge_amt) AS total_per_cat
                FROM charge
                GROUP BY category_no
            )
        )
) AS filtered_categories;
"""

In [38]:
db_file = 'local_credit_database.db'
result = execute_sql_query(db_file, query)
print(result)

   num_categories_above_avg_charge
0                                5
